# Toy Semantic Segmentation Comparison

This notebook runs the existing toy DINO semantic segmentation model on the full-model split directory layout. It uses the same wrapper functions as the sbatch script so notebook and batch behavior stay aligned.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
LFM_ROOT = NOTEBOOK_DIR.parents[1]

if str(LFM_ROOT) not in sys.path:
    sys.path.insert(0, str(LFM_ROOT))

print("Notebook directory:", NOTEBOOK_DIR)
print("LFM root:", LFM_ROOT)

In [ ]:
from argparse import Namespace

from lightning.pytorch import seed_everything

from lfm.full_model.utils import create_timestamped_output_dir
from toy_sem_seg_comparison import (
    build_config,
    create_datamodule,
    create_lightning_module,
    create_model,
    create_trainer,
    save_config,
    validate_data_paths,
)

## Config

In [ ]:
# Set this to the source data directory containing train/val/test split folders.
# The notebook will create NOTEBOOK_DIR / "data" as a symlink to this path.
SIMLINK_DEST = None  # Example: Path("/explore/nobackup/projects/lfm/model_inputs/my_sem_seg_split")

DATA_SYMLINK = NOTEBOOK_DIR / "data"

if SIMLINK_DEST is not None:
    source = Path(SIMLINK_DEST).expanduser().resolve()
    if not source.exists():
        raise FileNotFoundError(f"SIMLINK_DEST does not exist: {source}")
    if not source.is_dir():
        raise NotADirectoryError(f"SIMLINK_DEST must be a directory: {source}")

    if DATA_SYMLINK.is_symlink():
        current_target = DATA_SYMLINK.resolve()
        if current_target != source:
            raise FileExistsError(
                f"{DATA_SYMLINK} already points to {current_target}, not SIMLINK_DEST {source}. "
                "Remove or update the symlink explicitly before continuing."
            )
        print(f"Symlink created successfully: {DATA_SYMLINK} -> {source}")
    elif DATA_SYMLINK.exists():
        raise FileExistsError(
            f"{DATA_SYMLINK} already exists and is not a symlink. Move it before creating the data symlink."
        )
    else:
        DATA_SYMLINK.symlink_to(source, target_is_directory=True)
        print(f"Symlink created successfully: {DATA_SYMLINK} -> {source}")
else:
    print("SIMLINK_DEST is None; leaving ./data unchanged.")

In [ ]:
args = Namespace(
    data_root=None,  # Defaults to notebooks/full_model/data
    base_output_dir=None,  # Defaults to notebooks/full_model/outputs/toy_sem_seg_comparison
    dino_checkpoint=None,  # Uses the default checkpoint path in sseg_model.py
    band_filter=[0, 1, 2, 3, 4, 5, 6],
    target_size=256,
    spatial_transform="crop",
    batch_size=16,
    num_workers=10,
    max_epochs=100,
    learning_rate=5e-5,
    weight_decay=1e-3,
    loss_type="focal_dice",
    freeze_encoder=False,
    seed=42,
    no_fit=False,
)

config = build_config(args)
validate_data_paths(config)

print("Data root:", config.data_root)
print("Base output dir:", config.base_output_dir)
print("Band filter:", config.band_filter)
print("Target size:", config.target_size)
print("Spatial transform:", config.spatial_transform)
print("Normalize inputs:", config.normalize_inputs)
print("Max epochs:", config.max_epochs)

## Output Directory

In [ ]:
output_dir = create_timestamped_output_dir(config.base_output_dir)
save_config(config, output_dir)
print("Output dir:", output_dir)

## DataModule

In [ ]:
seed_everything(config.seed)
datamodule = create_datamodule(config, output_dir)

if datamodule.weight_assignments is None:
    raise RuntimeError("DataModule did not create weight assignments.")

print("Weight assignments:", datamodule.weight_assignments)

## Model

In [ ]:
model = create_model(config, datamodule.weight_assignments)
task = create_lightning_module(config, model)

print(type(model))
print(type(task))

## Trainer

In [ ]:
trainer = create_trainer(config, output_dir)

## Fit

In [ ]:
# Set args.max_epochs = 1 in the config cell for a smoke test.
if args.no_fit:
    print("Skipping trainer.fit() because args.no_fit is True.")
else:
    trainer.fit(task, datamodule=datamodule)

## Test Best Checkpoint

In [ ]:
if args.no_fit:
    print("Skipping trainer.test() because args.no_fit is True.")
else:
    trainer.test(task, datamodule=datamodule, ckpt_path="best")